# Régression de consommation finale — modèle final géographique v1

Cible : **conso_5_usages_ef, kWh/an**, sans coût ni étiquette en entrée. Les sept caractéristiques sont communes à la classification. Les corrections d'altitude de Nouhayla sont intégrées. Random Forest de 100 arbres, profondeur 10, min_samples_split=20, graine 42, deux threads. Réglages conservés sans nouvelle recherche, entraînement final sur les 453 680 lignes entraînement + validation.

Évaluation sur les 107 666 lignes réservées après gel des choix. **Lot interne à des données déjà exploitées : ce n'est pas un test externe indépendant.** Le notebook affiche les résultats figés et n'entraîne ni ne réévalue les modèles. La version précédente est conservée dans archives/.


In [1]:
from pathlib import Path
import os
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "MLmodels").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Ouvrez le notebook depuis le dépôt App_Enedis.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.resource_limits import configure
configure()
import pandas as pd
import json
from src.final_model_loader import load_final_bundle
bundle = load_final_bundle()
print("Modèles finaux rechargés ; prédictions de contrôle identiques.")
print("Aucun entraînement ni nouvelle évaluation du lot réservé.")


Modèles finaux rechargés ; prédictions de contrôle identiques.
Aucun entraînement ni nouvelle évaluation du lot réservé.


In [2]:
result = bundle["results"]
print("LOT RÉSERVÉ INTERNE — résultats sauvegardés")
print("MAE (kWh/an) :", result["regression"]["mae"])
print("RMSE (kWh/an) :", result["regression"]["rmse"])
print("R² :", result["regression"]["r2"])
print("Référence : médiane de l'entraînement + validation =", result["baseline"]["regression_median_kwh"])
display(pd.DataFrame([{"Modèle": "Random Forest", **result["regression"]},
                      {"Modèle": "Médiane constante", **result["baseline"]["regression"]}]))


LOT RÉSERVÉ INTERNE — résultats sauvegardés
MAE (kWh/an) : 1814.684644616723
RMSE (kWh/an) : 2957.205239411379
R² : 0.6530803007309904
Référence : médiane de l'entraînement + validation = 4754.0
              Modèle    rows          mae         rmse        r2
0      Random Forest  107666  1814.684645  2957.205239  0.653080
1  Médiane constante  107666  3505.669315  5142.884805 -0.049252


In [3]:
display(pd.DataFrame([{ "Population": source, **values["regression"]}
                      for source, values in result["by_source"].items()]))
display(pd.DataFrame(result["regression_by_dpe"]).T)


  Population   rows          mae         rmse        r2
0   existant  57331  2802.285962  3934.157946  0.467324
1       neuf  50335   689.817829  1037.680098  0.502414
      rows          mae         rmse        r2
A  29227.0   829.524495  1423.454729 -1.360913
B  13666.0  1952.835443  3402.257873 -2.709058
C  39536.0  2143.475232  3283.930294 -0.096172
D  16846.0  1956.198114  2708.077243  0.734320
E   6334.0  2660.939762  3689.505403  0.675534
F   1590.0  4264.696963  5563.175976  0.449275
G    467.0  6668.232376  9114.936811  0.338756


In [4]:
# Analyse des prédictions déjà conservées ; aucun nouvel appel au modèle sur le réservé.
path = ROOT / result["predictions_path"]
if path.is_file():
    predictions = pd.read_csv(path)
    residuals = predictions.true_consumption - predictions.predicted_consumption
    import matplotlib.pyplot as plt
    sample = predictions.sample(n=min(3000, len(predictions)), random_state=42)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(residuals, bins=60)
    axes[0].set_xlabel("Consommation réelle moins prédite (kWh/an)")
    axes[0].set_title("Résidus conservés — lot interne")
    axes[1].scatter(sample.true_consumption, sample.predicted_consumption, s=5, alpha=0.25)
    axes[1].set_xlabel("Consommation réelle (kWh/an)")
    axes[1].set_ylabel("Consommation prédite (kWh/an)")
    plt.tight_layout()
    plt.show()
else:
    print("Prédictions détaillées locales absentes ; les métriques et modèles restent disponibles.")


In [5]:
print("Variables :", bundle["variables"]["regression"]["features"])
print("Cible :", bundle["variables"]["regression"]["target"])
print("Unité :", bundle["variables"]["regression"]["unit"])
print("Paramètres :", bundle["variables"]["regression"]["model_parameters"])
print("Modèle : MLmodels/final_geo_v1/regression.joblib")


Variables : ['surface_habitable_logement', 'nombre_niveau_logement', 'age_batiment', 'altitude_moyenne', 'type_energie_principale_chauffage', 'type_batiment', 'zone_climatique']
Cible : conso_5_usages_ef
Unité : kWh/an, énergie finale cinq usages
Paramètres : {'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 10, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 20, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': 2, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
Modèle : MLmodels/final_geo_v1/regression.joblib


L'erreur est plus élevée dans l'existant que dans le neuf. La MAE n'est pas un intervalle de confiance individuel. Les anciens scores (MAE 2 121,45 ; R² 0,6932) concernent un autre CSV et un autre découpage ; leur différence avec les scores présents ne démontre pas un progrès du modèle. Une évaluation externe et une préparation ajustée uniquement sur l'entraînement restent à prévoir. Aucun réglage n'a été modifié après observation du lot réservé.
